
## Synthetic Light Curve Pipeline

This notebook generates a synthetic photometry light curve produced by the YMDF model
(Mamonova et al 2025 doi:[10.1051/0004-6361/202554614];
Mamonova et al.2026 doi:[10.1051/0004-6361/202556844])
This serves as a validation test: the synthetic flares in the synthetic light curve can be compared against the observed FFDs.


## Synthetic Light Curve — YMDF Model

Generates a synthetic spectral time series using the Young M Dwarf Flare (YMDF) model.
The pipeline proceeds in three stages:

1. **Flare frequency distribution fitting** — the observed TESS flare catalogue is fitted
   with a piecewise broken power law using maximum likelihood estimation (MMLE) and
   least-squares fitting, with MCMC-sampled synthetic populations validated against
   observations via a two-sample KS test
2. **Flare sequence generation** — a synthetic flare sequence is drawn from the fitted
   broken power law over a 10-day baseline
3. **Spectral time series construction** — each synthetic flare event is assigned a
   spectrally resolved flux contribution via `variableSpectraOneFlare`, and the resulting
   spectral time series is split into segments for downstream atmospheric chemistry modelling


In [ ]:
from astropy import units as u
from astropy import table
from matplotlib import pyplot as plt
import numpy as np
from astropy import constants
import pandas
from phab.ymdf.model import (
    variableSpectraOneFlare,
    variableSpectraOneFlareObs,
    calculateEDfromTESS,
    calculateEDFUVtoTESS,
    sampleBrokenPowerLaw,
    brokenPowerLaw,
    generateSimulatedData
)
from phab.ymdf.flare_finder import (
    findFlares,
    findGaps,
    sampleFlareRecovery,
    characterizeFlares,
    fitsToPandas,
    sigmaClip,
    addPaddingToList,
    detrendSavGol,
    luminosityQuiescent,
    fluxQuiescent
)


### Stellar Parameters

Stellar radius, effective temperature, and distance are pulled from the sample catalogue.
The projected stellar surface area and the distance-to-radius normalisation factor are
pre-computed here as they are required by the spectral flux scaling throughout the pipeline.


In [ ]:
star = "AU Mic"

R_star    = enriched.at[star, "radius_check"]
star_teff = enriched.at[star, "teff_check"]
star_dist = enriched.at[star, "Distance"]

area  = (numpy.pi * (R_star * constants.R_sun.to("cm"))**2).value
denom = ((star_dist * constants.pc.to("cm")) / (constants.R_sun.to("cm") * R_star))**2
denom = denom.value

print(f"R_star:    {R_star} R_sun")
print(f"T_eff:     {star_teff} K")
print(f"Distance:  {star_dist} pc")
print(f"Area:      {area:.4e} cm^2")
print(f"Denom:     {denom:.4e}")


### Load YMDF Spectral Models and Observed Flare Catalogue

The pre-computed YMDF spectral model grid is loaded from a pickle file. Each row
corresponds to a model spectrum with columns `out1` for the m2F12-37-2.5, `out2` for the cF13-500-3, and `flux_zero_model` for the quiescent flux (Kowalski A. F. et al. (2024), doi:[10.3847/1538-4357/ad4148],
indexed by wavelength in nm (range 1.0-5499.0). The observed TESS flare catalogue with corrected equivalent
durations and quiescent luminosities is also loaded here.


In [ ]:
models = pandas.read_pickle("./data/models.pkl")
# flux columns in "erg s-1 cm-2 nm-1";  wave in nm

flares_events_t = pandas.read_pickle(
    f"./{star}_flares_checked.pkl"
).reset_index(drop=True)

luminosityQ = flares_events_t.at[0, "luminosity quiscent"]
ed_tess     = [flares_events_t.at[i, "ed_corr"] for i in flares_events_t.index]

print(flares_events_t)
print(f"Quiescent luminosity: {luminosityQ:.4e} erg/s")
print(f"Number of observed flares: {len(ed_tess)}")


### Build Observed Flare Frequency Distribution

The corrected equivalent durations are sorted in descending order and converted to
cumulative flare frequencies by dividing by the total observation time `Tobs`.
A secondary x-axis showing flare energy in erg is added via the quiescent luminosity.


In [ ]:
Tobs   = 51.465710503720175 + (39784.76) / 24 / 60 / 60 # example for AU Mic observed sectors 1 and 27  in TESS and COS in days

ed_obs = numpy.sort(numpy.array(ed_tess))[::-1]
ed_obs = ed_obs[~numpy.isnan(ed_obs)]
freq   = numpy.arange(1, len(ed_obs) + 1) / Tobs

tog = pandas.DataFrame({
    "ed_obs" : ed_obs,
    "freq"   : freq,
    "energy" : luminosityQ * ed_obs
})

ed_tess_sorted = numpy.sort(ed_tess)[::-1]
freq_tess      = numpy.arange(1, len(ed_tess_sorted) + 1) / Tobs

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(ed_tess_sorted, freq_tess, c="tab:green", zorder=3, edgecolor="green", label="TESS (uncorrected)")
ax.scatter(ed_obs,         freq,      c="tab:red",   zorder=3, edgecolor="red",   label="TESS (corrected)")

ax2 = ax.secondary_xaxis("top", functions=(lambda x: x * luminosityQ, lambda x: x / luminosityQ))
ax2.set_xlabel("Flare energy [erg]", fontsize=12, labelpad=14)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Equivalent duration [s]", fontsize=12)
ax.set_ylabel("Cumulative frequency [day$^{-1}$]", fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

print(f"ED range: {ed_obs.min():.4f} – {ed_obs.max():.4f} s")


### Power Law Fitting

The flare frequency distribution is fitted with a piecewise broken power law using
two sequential steps:
1. An initial single power law index `alpha` is estimated via maximum likelihood
   estimation (MMLE)
2. The normalisation `beta` is derived analytically from `alpha`
3. A broken power law (`BrokenPowerLaw1D`) is then fitted using Levenberg–Marquardt
   least squares `LevMarLSQFitter`, or `SimplexLSQFitter`as a fallback if the primary fitter fails


In [ ]:
def fit_beta_to_powerlaw(x0, ed, freq, alpha, alpha_err):
    '''Fit beta via linear least squares to a power
    law with given alpha using the cumulative
    FFD. Estimate uncertainty using jackknife algorithm.

    Parameters:
    -----------
    mode : str
        ED or energy will set the starting value for the
        least square minimization
    x0starts : dict
        start values for LSQ fitting

    Return:
    -------
    _beta, beta, beta_err -  array, float, float
        jackknife sample of beta values, mean beta, beta uncertainty
    '''
    def LSQ(x0, ed, freq, alpha):
        zw = ((x0 /
               (numpy.power(ed, alpha - 1.) * (alpha - 1.)) - freq)**2).sum()
        return numpy.sqrt(zw)

    N = len(ed)
    if N == 0:
        raise ValueError('No data.')

    # jackknife uncertainty
    _beta = numpy.array([fmin(LSQ, x0,
                              args=(numpy.delete(ed, i),
                                    numpy.delete(freq, i),
                                    alpha),
                              disp=0)[0] for i in range(N)])

    # cumulative beta = beta_cum
    beta = _beta.mean()
    beta_err = numpy.sqrt((N - 1) / N * ((_beta - beta)**2).sum())

    # propagate errors on alpha to beta
    beta_err = (numpy.sqrt(beta_err**2 * (alpha - 1.)**2 +
                           beta**2 * alpha_err**2))

    # set attributes


    return beta, beta_err

In [ ]:
from astropy.modeling import models as astropy_models, fitting
from astropy.modeling.models import BrokenPowerLaw1D

alpha_ini, alpha_ini_error = fit_mmle_powerlaw(ed_obs)
print(f"alpha = {alpha_ini:.4f} +/- {alpha_ini_error:.4f}")

x0 = 1.
beta_prior, beta_err = fit_beta_to_powerlaw(x0, ed_obs, freq, alpha_ini, alpha_ini_error)
print(f"beta  = {beta_prior:.4f} +/- {beta_err:.4f}")

model      = BrokenPowerLaw1D(amplitude=beta_prior, x_break=10., alpha_1=0.38, alpha_2=0.89) #empirically found coefficients, see Mamonova et al 2025 doi:[10.1051/0004-6361/202554614]
fit        = None
fitted_model = None

try:
    fit          = fitting.LevMarLSQFitter(calc_uncertainties=True)
    fitted_model = fit(model, tog["ed_obs"].values, tog["freq"].values)
except Exception as ex:
    print(f"LevMarLSQFitter failed: {ex}")

if fitted_model is None:
    try:
        fit          = fitting.SimplexLSQFitter()
        fitted_model = fit(model, tog["ed_obs"].values, tog["freq"].values)
        print("Fitted with SimplexLSQFitter")
    except Exception as ex:
        print(f"SimplexLSQFitter failed: {ex}")

if fit is not None and fitted_model is not None:
    try:
        uncertainties = numpy.sqrt(numpy.diag(fit.fit_info["param_cov"]))
        print(f"Parameter uncertainties: {uncertainties}")
    except Exception as ex:
        print(f"Could not extract uncertainties: {ex}")
    print(fitted_model)

### MCMC Synthetic Population Validation

Synthetic flare populations are drawn from the fitted broken power law using
`generateSimulatedData`(For details see Mamonova et al.2026 doi:[10.1051/0004-6361/202556844]) and compared to the observed distribution via a two-sample KS test. Only populations with KS p-value > 0.1 are accepted. Accepted populations are overplotted on the frequency distribution and saved to disk for downstream spectral time series generation.


In [ ]:
x_min      = tog["ed_obs"].min()
x_max      = tog["ed_obs"].max()
EDmin      = x_min
EDmax      = x_max
EDmin_pop  = x_min / 10.
EDmax_pop  = x_max * 10.
xbreak     = fitted_model.x_break.value
a1         = fitted_model.alpha_1.value + 1.
a2         = fitted_model.alpha_2.value + 1.
Tprime     = 51.465710503720175 * 24 * 60 * 60 # TESS sectors only
interval   = 20
T0         = 0.0
start_seed = 130
number     = 499
array_seeds = numpy.arange(start_seed, start_seed + number + 1, dtype=int)

flaresperday = beta_prior * numpy.power(EDmin_pop, -(a1 - 1.)) / (a1 - 1.)
print(f"Expected flares per day: {flaresperday:.4f}")
print(f"ED range: {EDmin:.4f} – {EDmax:.4f} s")

w, i        = 0, 0
which_seed  = []
p_value     = 0.0

while p_value < 0.1:
    seed_0       = array_seeds[i]
    syntetic_eds = generateSimulatedData(
        flaresperday, Tprime, interval, beta_prior,
        a1, a2, xbreak, None,
        EDmin_pop, EDmax_pop,
        T0, law="piecewise"
    )

    syntetic_eds1 = syntetic_eds.query("ED >= @EDmin and ED <= @EDmax")
    ed_te         = numpy.sort(syntetic_eds1["ED"].values)[::-1]
    freq_te       = numpy.arange(1, len(ed_te) + 1) / Tprime * 24 * 60 * 60

    statistic, p_value = stats.ks_2samp(ed_obs, ed_te)
    print(f"Seed {seed_0}: KS p-value = {p_value:.4f}, n_synth = {len(ed_te)}")

    if p_value > 0.1:
        ed_te_a  = numpy.sort(syntetic_eds["ED"].values)[::-1]
        freq_te_a = numpy.arange(1, len(ed_te_a) + 1) / Tprime * 24 * 60 * 60
        ax.scatter(ed_te,   freq_te,   alpha=0.5, zorder=1, color="aqua")
        ax.scatter(ed_te_a, freq_te_a, alpha=0.5, zorder=2, color="pink")
        which_seed.append(seed_0)
        syntetic_eds.to_pickle(f"./synt_eds_seed{seed_0}.pkl") # you need to produce at least as many files as you have segments
        syntetic_eds.to_csv(f"./synt_eds_seed{seed_0}.csv")
        w += 1
    i += 1

with PdfPages(f"./distributionWR_{star}.pdf") as pdf:
    pdf.savefig(fig, bbox_inches="tight")

print(f"Accepted populations: {w}, seeds: {which_seed}")


### Spectral Time Series Generation

The accepted synthetic flare sequence is split into `n_segments` segments of equal
duration (864000 s = 10 days each). For each segment, `variableSpectraOneFlare`
assigns a spectrally resolved flux contribution to each flare event using the
Mendoza temporal flare model. Duplicate time indices are removed and boundary rows are
added to ensure each segment spans exactly `[0, segment_duration]`. Each segment
is saved as a pickle file for input to the atmospheric chemistry pipeline.


In [ ]:
# for easch segment load the ED file
i=0# change in [0,1,2,3,4]
syntetic_eds = pandas.read_pickle(f"./synt_eds_seed{which_seed[i]}.pkl")

In [ ]:
n_segments       = 5
segment_duration = 864000.
serie            = 0# start with 0 and for the 360 days it will be up to 7
k                = n_segments * serie
boundaries       = numpy.linspace(0., segment_duration * n_segments, n_segments + 1)
c                = 5.03

for i in range(n_segments):


    seg = syntetic_eds[
        (syntetic_eds["tstart"] >= boundaries[i]) &
        (syntetic_eds["tstart"] < boundaries[i + 1] if i < n_segments - 1 else syntetic_eds["tstart"] <= boundaries[i + 1])
        ]

    syntetic_spectre = pandas.DataFrame()
    for j in range(len(seg)):
        ED     = seg.iloc[j]["ED"]
        tstart = seg.iloc[j]["tstart"]
        oneFlare = variableSpectraOneFlare(
            c, ED, tstart, models,
            time=None,
            waverange=[600., 1000.],
            interval=20,
            model="Mendoza",
            seed=None,
            duration=None,
            tpeak_i=None
        )
        syntetic_spectre = pandas.concat([syntetic_spectre, oneFlare])

    syntetic_spectre = syntetic_spectre[~syntetic_spectre.index.duplicated(keep="first")]
    syntetic_spectre = syntetic_spectre.copy()
    syntetic_spectre.index = syntetic_spectre.index - segment_duration * i

    syntetic_spectre.loc[0.0]              = syntetic_spectre.iloc[0].copy()
    syntetic_spectre.loc[segment_duration] = syntetic_spectre.iloc[-1].copy()
    syntetic_spectre = syntetic_spectre.sort_index()
    syntetic_spectre = syntetic_spectre[syntetic_spectre.index <= segment_duration]

    outfile = f"./YMDF_sflux_timesteps_20secH-{k + i + 1}"
    syntetic_spectre.to_pickle(f"{outfile}.pkl")

    print(f"Segment {i+1}: {len(syntetic_spectre)} time steps, saved to {outfile}.pkl")
